In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# Load the dataset
df = pd.read_csv('d:\\Codes\\Patient_Readmission_Prediction\\data\\processed\\diabetic_data_cleaned.csv') 

In [4]:
df['total_visits'] = (df['number_outpatient'] + df['number_emergency'] + df['number_inpatient'])
df[['total_visits']].describe()

,total_visits
count,71518.000000
mean,0.932017
std,1.860583
min,0.000000
25%,0.000000
50%,0.000000
75%,1.000000
max,61.000000


This is the feature 1 that is the total prior visits of the patients.Patients who use hospitals frequently (high utilizers) tend to have complex, poorly-controlled conditions which might lead to the case of readmitting.

In [5]:
#feature 2: Was treatment changed during the visit?
df['treatment_changed'] = (df['change'] == 'Ch').astype(int)
df['treatment_changed'].value_counts()

treatment_changed
0    39510
1    32008
Name: count, dtype: int64

If doctors changed the medication prescribed, it signals the patient's condition was not well-controlled, which leads to high chances of readmission.

In [6]:
#feature 3: ICD-9 primary diagnosis category
def map_icd_to_category(icd):
    try:
        code = float(icd)
        if 390 <= code < 460 or code == 785:
            return 'Circulatory'
        if 460 <= code < 520 or code == 786:
            return 'Respiratory'
        if 520 <= code < 580 or code == 787:
            return 'Digestive'
        if str(icd).startswith('250'):
            return 'Diabetes'
        if 800 <= code < 1000:
            return 'Injury'
        if 710 <= code < 740:
            return 'Musculoskeletal'
        if 580 <= code < 630 :
            return 'Genitourinary'
        if 140 <= code < 240:
            return 'Neoplasms'
        return 'Other'
    except:
        return 'Other'

In [7]:
df['diagl_category'] = df['diag_1'].apply(map_icd_to_category)
df['diagl_category'].value_counts()

diagl_category
Circulatory        21349
Other              12800
Respiratory         9872
Digestive           6600
Diabetes            5690
Injury              4980
Musculoskeletal     3894
Genitourinary       3605
Neoplasms           2728
Name: count, dtype: int64

The raw data codes are meaningless to a model like '460','520' it does not know that it implies, hence by grouping into categories we give the model medically meaningful signal.

In [10]:
#feature 4: lab and medication intensity
df['lab_med_intensity'] = df['num_lab_procedures'] / (df['time_in_hospital'] + 1)
df['medication_intensity'] = df['num_medications'] / (df['time_in_hospital'] + 1)

Here high intensity represents more complex paitent which means that readmission might occur.

In [ ]:
new_features = [
    'total_visits',
    'treatment_changed',
    'diagl_category',
    'lab_med_intensity',
    'medication_intensity'
]

df[new_features].head()

,total_visits,treatment_changed,diagl_category,lab_med_intensity,medication_intensity
0,0,0,Diabetes,20.500000,0.500000
1,0,1,Other,14.750000,4.500000
2,3,0,Other,3.666667,4.333333
3,0,1,Other,14.666667,5.333333
4,0,1,Neoplasms,25.500000,4.000000


In [17]:
df.columns.tolist()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 71518 entries, 0 to 71517
Data columns (total 51 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   patient_nbr               71518 non-null  int64  
 1   race                      69640 non-null  str    
 2   gender                    71518 non-null  str    
 3   admission_type_id         71518 non-null  int64  
 4   discharge_disposition_id  71518 non-null  int64  
 5   admission_source_id       71518 non-null  int64  
 6   time_in_hospital          71518 non-null  int64  
 7   num_lab_procedures        71518 non-null  int64  
 8   num_procedures            71518 non-null  int64  
 9   num_medications           71518 non-null  int64  
 10  number_outpatient         71518 non-null  int64  
 11  number_emergency          71518 non-null  int64  
 12  number_inpatient          71518 non-null  int64  
 13  diag_1                    71501 non-null  str    
 14  diag_2           